## K-Fold Cross-Validation Summary (`ep200` epoch-budget runs)

Summarises the 5-fold CV runs trained for a fixed **epoch** budget (`--epochs 200`,
eval every 10 epochs → `ee10_0`) rather than a fixed iteration budget.

Since `max_iter = epochs * ceil(num_train/batch)` and fold train-set sizes differ
slightly, folds end with different `max_iter` (`mi6600` for folds 0–3, `mi6800` for
fold 4) and land their 20 eval checkpoints at different iteration numbers. The notebook
therefore:

- groups folds by a key that **ignores the per-fold `mi####` token** (so all 5 folds
  collapse into one group), and
- aligns per-fold AP curves against **epochs**, not raw iterations.

For each group it extracts the **best-epoch** metrics per fold (highest `segm/AP`) and
reports mean ± 1 std across folds.

In [1]:
# Setup: imports + metric/parsing helpers for the ep200 epoch-budget summary.
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("output")


# Which eval dataset to summarise across folds.
# Fold-specific held-out validation is the right default for k-fold CV.
# Switch to "rob2pheno_val" to use the full validation set instead.
EVAL_DATASET = "rob2pheno_fold_val"


def load_metrics(
    run_dir: Path,
    eval_dataset: str = EVAL_DATASET,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Read metrics.json (JSONL) and split into (train_df, eval_df).

    Detectron2 prefixes eval keys with the registered dataset name when
    more than one eval dataset is present, e.g. "rob2pheno_fold_val/segm/AP".
    This loader strips that prefix so downstream code can use the short
    names ("segm/AP", "segm/AP50", ...).
    """
    records = []
    with (run_dir / "metrics.json").open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    train_rows = [r for r in records if "total_loss" in r]
    eval_rows  = [
        r for r in records
        if any(k == "segm/AP" or k.endswith("/segm/AP") for k in r)
    ]

    def _to_df(rows):
        df = pd.DataFrame(rows)
        if df.empty or "iteration" not in df.columns:
            return df
        return df.sort_values("iteration").reset_index(drop=True)

    train_df = _to_df(train_rows)
    eval_df  = _to_df(eval_rows)

    if not eval_df.empty:
        prefix = f"{eval_dataset}/"
        rename = {c: c[len(prefix):] for c in eval_df.columns if c.startswith(prefix)}
        if rename:
            keep_cols = ["iteration"] + list(rename.keys())
            keep_cols = [c for c in keep_cols if c in eval_df.columns]
            eval_df = eval_df[keep_cols].rename(columns=rename)
        # else: columns already use the short form (older runs) - leave as-is

    return train_df, eval_df


def iters_per_epoch(dirname: str) -> int | None:
    """Recover iters/epoch from the _mi#### and _ep### tokens train.py writes.

    train.py sets max_iter = epochs * ceil(num_train/batch), so
    iters_per_epoch = max_iter / epochs.  Under --epochs (ep200) the max_iter
    differs per fold (mi6600 vs mi6800), so we read it back per directory to
    convert iterations -> epochs.  Returns None if the tokens are absent.
    """
    mi = re.search(r"_mi(\d+)", dirname)
    ep = re.search(r"_ep(\d+)", dirname)
    if not (mi and ep):
        return None
    return round(int(mi.group(1)) / int(ep.group(1)))


def fold_variant_key(dirname: str) -> str:
    """Base variant name shared by all folds of one run.

    Strips _fold{i}of{k}_seed{s} AND the per-fold-varying _mi#### token: under
    the epoch budget each fold's max_iter differs (mi6600/mi6800), so it must be
    removed for all 5 folds to collapse to one group key.
    """
    key = re.sub(r"_fold\d+of\d+_seed\d+", "", dirname)
    key = re.sub(r"_mi\d+", "", key)
    return key


def fold_index(dirname: str) -> int:
    """Fold index parsed from a "..._fold2of5_..." directory name (-1 if absent)."""
    m = re.search(r"_fold(\d+)of\d+", dirname)
    return int(m.group(1)) if m else -1


# Auto-discover all ep200 fold directories in output/ (excludes the older
# fixed-iteration mi8000 CV runs).
fold_groups: dict[str, list[str]] = {}
for d in sorted(OUTPUT_DIR.iterdir()):
    if d.is_dir() and "_ep200" in d.name and re.search(r"_fold\d+of\d+", d.name):
        key = fold_variant_key(d.name)
        fold_groups.setdefault(key, []).append(d.name)

print("Discovered fold groups:")
for key, folds in fold_groups.items():
    print(f"  {key}: {len(folds)} folds")
    for f in folds:
        print(f"    - {f}")


Discovered fold groups:
  rgb_ep200_ee10_0_cos_frz_sz1280_swin_tiny: 5 folds
    - rgb_ep200_mi6600_ee10_0_cos_frz_sz1280_fold0of5_seed42_swin_tiny
    - rgb_ep200_mi6600_ee10_0_cos_frz_sz1280_fold1of5_seed42_swin_tiny
    - rgb_ep200_mi6600_ee10_0_cos_frz_sz1280_fold2of5_seed42_swin_tiny
    - rgb_ep200_mi6600_ee10_0_cos_frz_sz1280_fold3of5_seed42_swin_tiny
    - rgb_ep200_mi6800_ee10_0_cos_frz_sz1280_fold4of5_seed42_swin_tiny
  rgb_ep200_ee10_0_cos_sz1280_swin_tiny: 5 folds
    - rgb_ep200_mi6600_ee10_0_cos_sz1280_fold0of5_seed42_swin_tiny
    - rgb_ep200_mi6600_ee10_0_cos_sz1280_fold1of5_seed42_swin_tiny
    - rgb_ep200_mi6600_ee10_0_cos_sz1280_fold2of5_seed42_swin_tiny
    - rgb_ep200_mi6600_ee10_0_cos_sz1280_fold3of5_seed42_swin_tiny
    - rgb_ep200_mi6800_ee10_0_cos_sz1280_fold4of5_seed42_swin_tiny
  rgb_ep200_ee10_0_sz1280_swin_tiny: 5 folds
    - rgb_ep200_mi6600_ee10_0_sz1280_fold0of5_seed42_swin_tiny
    - rgb_ep200_mi6600_ee10_0_sz1280_fold1of5_seed42_swin_tiny
    - rgb_ep2

Runs are grouped by their shared hyper parameters. **SELECT** the group evaluated here.

In [ ]:
# Run groups, organised by shared training hyperparameters.
#
# Directory-name axes:
#   schedule : (default) WarmupMultiStepLR step-decay  |  cos = WarmupCosineLR
#   tuning   : (default) full fine-tune                |  frz = frozen backbone
#   dca lr   : dlr5_0 = --dca-lr-mult 5  (rgbd_dca* variants only)
#
# To pick a group: set SELECTED_RUNS to one of the dicts below. Within a group,
# comment out individual variants you don't want. The report at the bottom checks
# the selection against output/ and skips anything not yet trained.

# Group A - step-decay schedule, full fine-tune (no warmup ramp)
RUNS_STEP_FT = {
    "rgb_ep200_ee10_0_sz1280_swin_tiny":              "RGB",
    "rgbd_early_ep200_ee10_0_sz1280_swin_tiny":       "RGB-D Early",
    "rgbd_dca_iter0_ep200_ee10_0_sz1280_swin_tiny":   "RGB-D DCA (K=0)",
    "rgbd_dca_iter1_ep200_ee10_0_sz1280_swin_tiny":   "RGB-D DCA (K=1)",
    "rgbd_dca_iter2_ep200_ee10_0_sz1280_swin_tiny":   "RGB-D DCA (K=2)",
}

# Group B - cosine schedule, frozen backbone, dca-lr-mult 5
RUNS_COS_FRZ = {
    "rgb_ep200_ee10_0_cos_frz_sz1280_swin_tiny":                   "RGB",
    "rgbd_early_ep200_ee10_0_cos_frz_sz1280_swin_tiny":            "RGB-D Early",
    "rgbd_dca_iter0_ep200_ee10_0_cos_dlr5_0_frz_sz1280_swin_tiny": "RGB-D DCA (K=0)",
    "rgbd_dca_iter1_ep200_ee10_0_cos_dlr5_0_frz_sz1280_swin_tiny": "RGB-D DCA (K=1)",
    "rgbd_dca_iter2_ep200_ee10_0_cos_dlr5_0_frz_sz1280_swin_tiny": "RGB-D DCA (K=2)",
}

# Group C - cosine schedule, full fine-tune, dca-lr-mult 5 (main result)
RUNS_COS_FT = {
    "rgb_ep200_ee10_0_cos_sz1280_swin_tiny":                   "RGB",
    "rgbd_early_ep200_ee10_0_cos_sz1280_swin_tiny":            "RGB-D Early",
    "rgbd_dca_iter0_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny": "RGB-D DCA (K=0)",
    "rgbd_dca_iter1_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny": "RGB-D DCA (K=1)",
    "rgbd_dca_iter2_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny": "RGB-D DCA (K=2)",
}

# All groups, for the "unreferenced runs on disk" check below.
ALL_GROUPS = {
    "RUNS_STEP_FT": RUNS_STEP_FT,
    "RUNS_COS_FRZ": RUNS_COS_FRZ,
    "RUNS_COS_FT":  RUNS_COS_FT,
}

# Active group (swap to RUNS_STEP_FT / RUNS_COS_FRZ for the appendix groups):
SELECTED_RUNS = RUNS_COS_FT


# --- Resolve the active selection against what is actually in output/ --------
_discovered = dict(fold_groups)   # full set from the discovery cell above

_present = {k: v for k, v in SELECTED_RUNS.items() if k in _discovered}
_absent  = {k: v for k, v in SELECTED_RUNS.items() if k not in _discovered}

if _absent:
    print("NOT YET TRAINED (in SELECTED_RUNS but no output/ dir) - skipped:")
    for k, v in _absent.items():
        print(f"  [missing] {v:<24} <- {k}")
    print()

if not _present:
    raise KeyError(
        "None of the active SELECTED_RUNS keys were found on disk.\n\nAvailable groups:\n  "
        + "\n  ".join(_discovered)
    )

# LABELS maps the (present) group key -> display label used downstream.
LABELS = {k: (v if v else k) for k, v in _present.items()}

# Restrict fold_groups to the present selection, preserving SELECTED_RUNS order.
fold_groups = {k: _discovered[k] for k in _present}

print(f"Selected {len(fold_groups)} run(s) for visualisation:")
for key in fold_groups:
    print(f"  {LABELS[key]:<24} <- {key} ({len(fold_groups[key])} folds)")

# --- Coverage report: discovered runs not referenced by ANY group -----------
_referenced = {k for g in ALL_GROUPS.values() for k in g}
_unreferenced = [k for k in _discovered if k not in _referenced]
if _unreferenced:
    print("\nON DISK but not in any group above (add them if you want them):")
    for k in _unreferenced:
        print(f"  [unused] {k}")

# --- Per-group missing report: defined runs with no output/ dir yet ---------
print("\nGroup coverage (defined vs trained-on-disk):")
for gname, group in ALL_GROUPS.items():
    miss = [k for k in group if k not in _discovered]
    status = "complete" if not miss else f"{len(miss)} missing"
    print(f"  {gname:<14} {len(group) - len(miss)}/{len(group)} trained  ({status})")
    for k in miss:
        print(f"      - {group[k]:<24} {k}")


Selected 5 run(s) for visualisation:
  RGB                      <- rgb_ep200_ee10_0_cos_sz1280_swin_tiny (5 folds)
  RGB-D Early              <- rgbd_early_ep200_ee10_0_cos_sz1280_swin_tiny (5 folds)
  RGB-D DCA (K=0)          <- rgbd_dca_iter0_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny (5 folds)
  RGB-D DCA (K=1)          <- rgbd_dca_iter1_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny (5 folds)
  RGB-D DCA (K=2)          <- rgbd_dca_iter2_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny (5 folds)

Group coverage (defined vs trained-on-disk):
  RUNS_STEP_FT   5/5 trained  (complete)
  RUNS_COS_FRZ   5/5 trained  (complete)
  RUNS_COS_FT    5/5 trained  (complete)


In [3]:
METRIC_COLS_CV = [
    "segm/AP", "segm/AP50", "segm/AP75",
    "segm/AP-redfruit", "segm/AP-greenfruit",
]

# Display labels for the metric columns (edit the values to taste).
METRIC_LABELS = {
    "segm/AP":            "AP",
    "segm/AP50":          "AP50",
    "segm/AP75":          "AP75",
    "segm/AP-redfruit":   "AP (red)",
    "segm/AP-greenfruit": "AP (green)",
}

cv_best: dict[str, pd.DataFrame] = {}   # variant -> DataFrame(fold x metric)
cv_eval: dict[str, list[pd.DataFrame]] = {}  # variant -> list of eval_df per fold

for variant, fold_dirs in fold_groups.items():
    best_rows = []
    eval_curves = []
    for dirname in sorted(fold_dirs):
        run_path = OUTPUT_DIR / dirname
        if not (run_path / "metrics.json").exists():
            print(f"  WARNING: missing metrics.json for {dirname}")
            continue
        _, eval_df = load_metrics(run_path)
        if eval_df.empty or "segm/AP" not in eval_df.columns:
            print(f"  WARNING: no segm/AP rows in {dirname}")
            continue

        # Under the epoch budget each fold's max_iter differs, so its eval
        # checkpoints land on different iterations. Attach an epoch column
        # (iteration / iters_per_epoch) so curves can be aligned across folds.
        ipe = iters_per_epoch(dirname)
        if ipe:
            eval_df = eval_df.assign(
                epoch=(eval_df["iteration"] / ipe).round().astype(int)
            )

        best_idx = eval_df["segm/AP"].idxmax()
        keep_cols = [c for c in METRIC_COLS_CV if c in eval_df.columns]
        extra = ["iteration"] + (["epoch"] if "epoch" in eval_df.columns else [])
        best = eval_df.loc[best_idx, keep_cols + extra].to_dict()
        best["fold"] = dirname
        best_rows.append(best)
        eval_curves.append(eval_df)

    if not best_rows:
        print(f"{variant}: no valid folds, skipping")
        continue
    cv_best[variant] = pd.DataFrame(best_rows).set_index("fold")
    cv_eval[variant] = eval_curves
    print(f"{variant}: {len(best_rows)} folds loaded")


rgb_ep200_ee10_0_cos_sz1280_swin_tiny: 5 folds loaded
rgbd_early_ep200_ee10_0_cos_sz1280_swin_tiny: 5 folds loaded
rgbd_dca_iter0_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny: 5 folds loaded
rgbd_dca_iter1_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny: 5 folds loaded
rgbd_dca_iter2_ep200_ee10_0_cos_dlr5_0_sz1280_swin_tiny: 5 folds loaded


In [4]:
if not cv_best:
    raise RuntimeError("cv_best is empty - run the previous cell first or check that output/ contains fold runs")

rows = []
for variant, best_df in cv_best.items():
    row = {"variant": variant}
    for col in METRIC_COLS_CV:
        if col in best_df.columns:
            row[f"{col}_mean"] = best_df[col].mean()
            row[f"{col}_std"]  = best_df[col].std(ddof=1)
    rows.append(row)

cv_summary = pd.DataFrame(rows).set_index("variant")

# Build a "mean +/- std" display table, using friendly metric + run labels.
display_df = pd.DataFrame(index=cv_summary.index)
for col in METRIC_COLS_CV:
    m_col, s_col = f"{col}_mean", f"{col}_std"
    if m_col not in cv_summary.columns:
        continue
    display_df[METRIC_LABELS.get(col, col)] = cv_summary.apply(
        lambda r, m=m_col, s=s_col: f"{r[m]:.2f} ± {r[s]:.2f}", axis=1
    )
# Relabel the rows (runs) with the selected display labels.
display_df.index = [LABELS.get(v, v) for v in display_df.index]
display_df.index.name = "Variant"
display_df


,AP,AP50,AP75,AP (red),AP (green)
Variant,,,,,
RGB,70.22 ± 1.84,94.93 ± 0.78,82.55 ± 1.18,70.84 ± 4.46,69.61 ± 2.89
RGB-D Early,67.37 ± 2.32,93.89 ± 2.33,78.67 ± 3.28,68.63 ± 3.79,66.11 ± 3.75
RGB-D DCA (K=0),68.19 ± 2.12,94.15 ± 2.52,79.80 ± 1.66,69.06 ± 3.85,67.33 ± 2.81
RGB-D DCA (K=1),67.43 ± 1.31,94.49 ± 1.58,77.95 ± 2.68,68.46 ± 3.42,66.40 ± 2.54
RGB-D DCA (K=2),67.48 ± 2.02,93.90 ± 1.35,77.43 ± 1.93,68.02 ± 4.18,66.95 ± 3.68


In [5]:
# Export display_df to LaTeX for Overleaf: an "overall" tabular (AP/AP50/AP75) and
# a "per-class" tabular (red/green), with the best (highest mean) entry per column
# bolded.  "±" -> $\pm$; booktabs rules need \usepackage{booktabs} in Overleaf.
OVERALL_COLS  = ["AP", "AP50", "AP75"]
PERCLASS_COLS = ["AP (red)", "AP (green)"]


def _cell_mean(cell: str) -> float:
    """Mean (the number before '±') of a 'mean ± std' display cell."""
    return float(cell.split("±")[0])


def _latex_tabular(cols: list[str]) -> str:
    # Highest-mean row per column -> bold.
    best = {c: display_df[c].map(_cell_mean).idxmax() for c in cols}
    out = [
        r"\begin{tabular}{l" + "c" * len(cols) + "}",
        r"\toprule",
        "Variant & " + " & ".join(cols) + r" \\",
        r"\midrule",
    ]
    for variant, row in display_df.iterrows():
        out.append(str(variant))
        for i, c in enumerate(cols):
            body = row[c].replace("±", r"$\pm$")
            if best[c] == variant:
                body = rf"\textbf{{{body}}}"
            end = r" \\" if i == len(cols) - 1 else ""
            out.append(f"& {body}{end}")
    out += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(out)


latex_str = (
    "% --- Overall metrics ---\n"
    + _latex_tabular(OVERALL_COLS)
    + "\n\n\\vspace{0.8em}\n\n"
    + "% --- Per-class metrics ---\n"
    + _latex_tabular(PERCLASS_COLS)
    + "\n\n\\end{table}"
)
print(latex_str)


% --- Overall metrics ---
\begin{tabular}{lccc}
\toprule
Variant & AP & AP50 & AP75 \\
\midrule
RGB
& \textbf{70.22 $\pm$ 1.84}
& \textbf{94.93 $\pm$ 0.78}
& \textbf{82.55 $\pm$ 1.18} \\
RGB-D Early
& 67.37 $\pm$ 2.32
& 93.89 $\pm$ 2.33
& 78.67 $\pm$ 3.28 \\
RGB-D DCA (K=0)
& 68.19 $\pm$ 2.12
& 94.15 $\pm$ 2.52
& 79.80 $\pm$ 1.66 \\
RGB-D DCA (K=1)
& 67.43 $\pm$ 1.31
& 94.49 $\pm$ 1.58
& 77.95 $\pm$ 2.68 \\
RGB-D DCA (K=2)
& 67.48 $\pm$ 2.02
& 93.90 $\pm$ 1.35
& 77.43 $\pm$ 1.93 \\
\bottomrule
\end{tabular}

\vspace{0.8em}

% --- Per-class metrics ---
\begin{tabular}{lcc}
\toprule
Variant & AP (red) & AP (green) \\
\midrule
RGB
& \textbf{70.84 $\pm$ 4.46}
& \textbf{69.61 $\pm$ 2.89} \\
RGB-D Early
& 68.63 $\pm$ 3.79
& 66.11 $\pm$ 3.75 \\
RGB-D DCA (K=0)
& 69.06 $\pm$ 3.85
& 67.33 $\pm$ 2.81 \\
RGB-D DCA (K=1)
& 68.46 $\pm$ 3.42
& 66.40 $\pm$ 2.54 \\
RGB-D DCA (K=2)
& 68.02 $\pm$ 4.18
& 66.95 $\pm$ 3.68 \\
\bottomrule
\end{tabular}

\end{table}


In [6]:
# Best EPOCH per fold (the eval checkpoint with highest segm/AP).
# Reported in epochs rather than raw iterations: under the epoch budget each fold's
# max_iter differs (mi6600/mi6800), so iteration numbers aren't comparable across
# folds, but the epoch axis (10..200) is.
best_epoch_rows = []
for variant, best_df in cv_best.items():
    row = {"variant": variant}
    # Prefer the precomputed epoch column; fall back to converting iteration.
    if "epoch" in best_df.columns:
        epochs = best_df["epoch"].astype(int)
    else:
        epochs = pd.Series(
            {fn: round(it / iters_per_epoch(fn))
             for fn, it in best_df["iteration"].items()}
        ).astype(int)
    # one column per fold index, ordered by fold name
    for fold_name, ep in epochs.items():
        idx = fold_index(fold_name)
        col = f"fold{idx}" if idx >= 0 else fold_name
        row[col] = ep
    row["mean"]   = epochs.mean()
    row["std"]    = epochs.std(ddof=1)
    best_epoch_rows.append(row)

best_epoch_df = pd.DataFrame(best_epoch_rows).set_index("variant")
# Sort fold columns naturally
fold_cols = sorted([c for c in best_epoch_df.columns if c.startswith("fold")],
                   key=lambda s: int(s.replace("fold", "")))
best_epoch_df = best_epoch_df[fold_cols + ["mean", "std"]]

# Relabel the rows (runs) with the selected display labels.
best_epoch_df.index = [LABELS.get(v, v) for v in best_epoch_df.index]
best_epoch_df.index.name = "run (best epoch)"

best_epoch_df.style.format({**{c: "{:.0f}" for c in fold_cols},
                            "mean": "{:.0f}", "std": "{:.0f}"})


,fold0,fold1,fold2,fold3,fold4,mean,std
run (best epoch),,,,,,,
RGB,40,40,30,30,40,36,5
RGB-D Early,90,30,100,70,60,70,27
RGB-D DCA (K=0),130,60,90,70,50,80,32
RGB-D DCA (K=1),130,90,90,120,60,98,28
RGB-D DCA (K=2),130,120,70,40,30,78,45


## 5.5 Paired statistical comparison

All five folds use the **same seed-42 partition** across every configuration, so the
per-fold `RGB − variant` differences are *paired* — the setup for a **paired t-test** /
**Wilcoxon signed-rank test**. For each fusion variant the cell reports the mean per-fold
AP gap, how many of the five folds RGB wins, and both p-values.

In [ ]:
# --- 5.5 Paired statistical comparison (RGB baseline vs each fusion variant) ---
# Folds share one seed-42 partition across variants, so per-fold differences are PAIRED.
# We report mean ΔAP, fold-win count, and paired t-test + Wilcoxon signed-rank p-values.

from scipy.stats import ttest_rel, wilcoxon

BASELINE = "RGB"

# Per-fold AP matrix aligned by fold index (rows=fold, cols=display variant name).
ap_by_variant = {}
for variant, best_df in cv_best.items():
    s = best_df["segm/AP"].copy()
    s.index = [fold_index(i) for i in s.index]
    ap_by_variant[LABELS.get(variant, variant)] = s.sort_index()
ap_mat = pd.DataFrame(ap_by_variant)

assert BASELINE in ap_mat.columns, f"{BASELINE!r} not among {list(ap_mat.columns)}"

stat_rows = []
for col in [c for c in ap_mat.columns if c != BASELINE]:
    paired = ap_mat[[BASELINE, col]].dropna()
    diff = paired[BASELINE] - paired[col]          # positive => RGB better
    n = len(diff)
    _, t_p = ttest_rel(paired[BASELINE], paired[col])
    try:
        _, w_p = wilcoxon(paired[BASELINE], paired[col])
    except ValueError:                              # all-zero diffs
        w_p = np.nan
    stat_rows.append({
        "Comparison": f"{BASELINE} vs {col}",
        "mean ΔAP": diff.mean(),
        "RGB wins": f"{int((diff > 0).sum())}/{n}",
        "paired-t p": t_p,
        "Wilcoxon p": w_p,
    })

stats_df = pd.DataFrame(stat_rows).set_index("Comparison")
print(stats_df.to_string(
    formatters={"mean ΔAP": "{:+.2f}".format,
                "paired-t p": "{:.3f}".format,
                "Wilcoxon p": "{:.3f}".format}))
stats_df


                       mean ΔAP RGB wins paired-t p Wilcoxon p
Comparison                                                    
RGB vs RGB-D Early        +2.85      5/5      0.018      0.062
RGB vs RGB-D DCA (K=0)    +2.03      5/5      0.023      0.062
RGB vs RGB-D DCA (K=1)    +2.80      5/5      0.001      0.062
RGB vs RGB-D DCA (K=2)    +2.74      5/5      0.002      0.062


,mean ΔAP,RGB wins,paired-t p,Wilcoxon p
Comparison,,,,
RGB vs RGB-D Early,2.850863,5/5,0.018061,0.0625
RGB vs RGB-D DCA (K=0),2.031128,5/5,0.022731,0.0625
RGB vs RGB-D DCA (K=1),2.797282,5/5,0.001431,0.0625
RGB vs RGB-D DCA (K=2),2.739516,5/5,0.001815,0.0625


## 5.6 Held-out 40-image set (independent check)

Re-evaluates the **same checkpoints** behind Table 5.1 — selected on in-fold validation —
on the common 40-image held-out set that was excluded from cross-validation entirely. If
the ranking here agrees with the CV table, the result is not an artefact of the fold
partition. Produces a `mean ± std` table plus a LaTeX export (`tab:heldout_results`).

In [8]:
# --- 5.6 Held-out 40-image set (independent check) ---
# Re-evaluate the SAME checkpoints used in Table 5.1 on the common held-out set
# (rob2pheno_val), which was excluded from the cross-validation entirely.
# Checkpoint selection stays on in-fold validation (cv_best) to avoid selecting on
# the held-out set; we only *read off* held-out metrics at that iteration.
HELDOUT_DATASET = "rob2pheno_val"

heldout_rows = []
for variant, fold_dirs in fold_groups.items():
    sel_iters = cv_best[variant]["iteration"].astype(int)  # index = fold dirname
    metric_vals: dict[str, list[float]] = {m: [] for m in METRIC_COLS_CV}
    for dirname in sorted(fold_dirs):
        run_path = OUTPUT_DIR / dirname
        if dirname not in sel_iters.index or not (run_path / "metrics.json").exists():
            continue
        _, eval_df = load_metrics(run_path, eval_dataset=HELDOUT_DATASET)
        if eval_df.empty or "segm/AP" not in eval_df.columns:
            print(f"  WARNING: no {HELDOUT_DATASET} segm/AP in {dirname}")
            continue
        eval_df = eval_df.dropna(subset=["segm/AP"]).set_index("iteration")
        it = sel_iters[dirname]
        if it not in eval_df.index:
            print(f"  WARNING: in-fold best iter {it} missing in held-out eval for {dirname}")
            continue
        for m in METRIC_COLS_CV:
            if m in eval_df.columns:
                metric_vals[m].append(float(eval_df.loc[it, m]))
    row = {"variant": variant}
    for m in METRIC_COLS_CV:
        vals = np.array(metric_vals[m], dtype=float)
        if len(vals):
            row[f"{m}_mean"] = vals.mean()
            row[f"{m}_std"] = vals.std(ddof=1)
    heldout_rows.append(row)

heldout_summary = pd.DataFrame(heldout_rows).set_index("variant")

heldout_display = pd.DataFrame(index=heldout_summary.index)
for col in METRIC_COLS_CV:
    mc, sc = f"{col}_mean", f"{col}_std"
    if mc in heldout_summary.columns:
        heldout_display[METRIC_LABELS.get(col, col)] = heldout_summary.apply(
            lambda r, m=mc, s=sc: f"{r[m]:.2f} ± {r[s]:.2f}", axis=1
        )
heldout_display.index = [LABELS.get(v, v) for v in heldout_display.index]
heldout_display.index.name = "Variant"

# LaTeX export (mirrors the cv table cell)
heldout_latex = heldout_display.map(lambda s: s.replace("±", r"$\pm$")).to_latex(
    escape=False,
    column_format="l" + "c" * heldout_display.shape[1],
    caption="Held-out 40-image set: segmentation AP at the in-fold-selected checkpoint "
            "(mean $\\pm$ 1 std across the 5 folds' models).",
    label="tab:heldout_results",
    position="ht",
)
print(heldout_latex)
heldout_display


\begin{table}[ht]
\caption{Held-out 40-image set: segmentation AP at the in-fold-selected checkpoint (mean $\pm$ 1 std across the 5 folds' models).}
\label{tab:heldout_results}
\begin{tabular}{lccccc}
\toprule
 & AP & AP50 & AP75 & AP (red) & AP (green) \\
Variant &  &  &  &  &  \\
\midrule
RGB & 67.20 $\pm$ 0.87 & 93.01 $\pm$ 1.17 & 78.14 $\pm$ 0.91 & 68.37 $\pm$ 1.44 & 66.02 $\pm$ 0.84 \\
RGB-D Early & 64.32 $\pm$ 0.84 & 91.95 $\pm$ 0.60 & 73.41 $\pm$ 1.03 & 66.06 $\pm$ 1.75 & 62.59 $\pm$ 0.41 \\
RGB-D DCA (K=0) & 64.50 $\pm$ 1.06 & 92.22 $\pm$ 1.18 & 73.47 $\pm$ 1.96 & 65.43 $\pm$ 1.44 & 63.57 $\pm$ 0.95 \\
RGB-D DCA (K=1) & 62.66 $\pm$ 1.11 & 91.00 $\pm$ 1.36 & 71.99 $\pm$ 1.98 & 63.94 $\pm$ 1.65 & 61.38 $\pm$ 0.99 \\
RGB-D DCA (K=2) & 64.52 $\pm$ 1.24 & 92.45 $\pm$ 1.00 & 74.59 $\pm$ 1.47 & 66.16 $\pm$ 1.14 & 62.87 $\pm$ 1.38 \\
\bottomrule
\end{tabular}
\end{table}



,AP,AP50,AP75,AP (red),AP (green)
Variant,,,,,
RGB,67.20 ± 0.87,93.01 ± 1.17,78.14 ± 0.91,68.37 ± 1.44,66.02 ± 0.84
RGB-D Early,64.32 ± 0.84,91.95 ± 0.60,73.41 ± 1.03,66.06 ± 1.75,62.59 ± 0.41
RGB-D DCA (K=0),64.50 ± 1.06,92.22 ± 1.18,73.47 ± 1.96,65.43 ± 1.44,63.57 ± 0.95
RGB-D DCA (K=1),62.66 ± 1.11,91.00 ± 1.36,71.99 ± 1.98,63.94 ± 1.65,61.38 ± 0.99
RGB-D DCA (K=2),64.52 ± 1.24,92.45 ± 1.00,74.59 ± 1.47,66.16 ± 1.14,62.87 ± 1.38
